In [1]:
import gzip
import json
import pandas as pd

from typing import Any

Parsing the mail file

In [2]:
df = pd.read_parquet("data_acl-ocl/acl-publication-info.74k.v2.parquet")

In [3]:
df.keys()

Index(['acl_id', 'abstract', 'full_text', 'corpus_paper_id', 'pdf_hash',
       'numcitedby', 'url', 'publisher', 'address', 'year', 'month',
       'booktitle', 'author', 'title', 'pages', 'doi', 'number', 'volume',
       'journal', 'editor', 'isbn', 'ENTRYTYPE', 'ID', 'language', 'note'],
      dtype='object')

In [4]:
df.head(1)

,acl_id,abstract,full_text,corpus_paper_id,pdf_hash,numcitedby,url,publisher,address,year,...,doi,number,volume,journal,editor,isbn,ENTRYTYPE,ID,language,note
0,O02-2002,There is a need to measure word similarity whe...,There is a need to measure word similarity whe...,18022704,0b09178ac8d17a92f16140365363d8df88c757d0,14,https://aclanthology.org/O02-2002,None,None,2002,...,None,None,None,None,None,None,inproceedings,chen-you-2002-study,None,None


In [5]:
df.drop(labels=["corpus_paper_id", "pdf_hash", "numcitedby", "url", "publisher", 
                "address", "year", "month", "booktitle", "author", "pages", "doi",
                "number", "volume", "journal", "editor", "isbn", "ENTRYTYPE", "ID", 
                "language", "note"], axis=1, inplace=True)

In [6]:
df.head(1)

,acl_id,abstract,full_text,title
0,O02-2002,There is a need to measure word similarity whe...,There is a need to measure word similarity whe...,A Study on Word Similarity using Context Vecto...


In [7]:
len(df)

73285

In [8]:
df.dropna(subset=["abstract"], inplace=True)

In [9]:
len(df)

67669

In [11]:
df.to_parquet("data_acl-ocl/acl-ocl_74k_v2_small.parquet", compression='gzip')

Attempting ot parse and format "full sections" (download data)

In [3]:
def load_pickle(file_path: str) -> Any:
    with gzip.open(file_path, "rb") as fp:
        data = pd.read_pickle(fp)
    return data

fs_data = load_pickle("data_acl-ocl/acl-publication-info.74k.v2.full-sections.pkl")

In [8]:
fs_data.keys()

Index(['acl_id', 'abstract', 'full_text', 'corpus_paper_id', 'pdf_hash',
       'numcitedby', 'url', 'publisher', 'address', 'year', 'month',
       'booktitle', 'author', 'title', 'pages', 'doi', 'number', 'volume',
       'journal', 'editor', 'isbn', 'ENTRYTYPE', 'ID', 'language', 'note',
       'json_contents'],
      dtype='object')

In [10]:
fs_data.drop(labels=["corpus_paper_id", "pdf_hash", "numcitedby", "url", "publisher", 
                     "address", "year", "month", "booktitle", "author", "pages", "doi",
                     "number", "volume", "journal", "editor", "isbn", "ENTRYTYPE", "ID", 
                     "language", "note"], axis=1, inplace=True)

In [21]:
len(fs_data)

67732

In [22]:
fs_data.dropna(subset=["abstract"], inplace=True)

In [23]:
len(fs_data)

63909

Properly parse the data in "json_contents"; "full_text" has all the content but formatting is not great.

In [94]:
def parse_sections(body_text: str) -> str:
    body_map = {}
    sec_ids = []
    sec_titles = []
    for chunk in body_text:
        # Handle the section numbers
        sec_num = chunk["sec_num"]
        sec_title = chunk["section"]

        # Fix formatting section number by removing trailing '.'
        if sec_num is not None and sec_num[-1]==".":
            sec_num = sec_num[:-1]
        # print(f"{sec_num} -- {sec_title}")

        if sec_num is None:
            if len(sec_ids) == 0:
                # Handle the introductions without section header
                sec_ids.append('0')
                sec_titles.append('Introduction')
                body_map['0'] = "Introduction\n\n"
            else:
                sec_num = sec_ids[-1]
        elif len(sec_num) == 1:
            # Section
            sec_ids.append(sec_num)
            sec_titles.append(sec_title)
        else:
            # Subsection etc. Save the title but under the correct section num.
            sec_num = sec_ids[-1]
            subsec_title = f"{chunk['sec_num']}: {chunk['section']}"
            if subsec_title not in body_map[sec_num]:
                body_map[sec_num] += f"\n{chunk['sec_num']}: {chunk['section']}\n\n"

        # If missing, initialise a new section block
        if sec_num not in body_map:
            body_map[sec_num] = f"SECTION {sec_num}: {sec_title}\n\n"
        
        # Just append the body text
        body_map[sec_num] += f"{chunk['text']}\n"

    print(body_map)
    return "\n".join([body_map[sec] for sec in body_map])

In [95]:
all_papers = {}

for idx, row in fs_data.iterrows():
    print(row["acl_id"])
    # Prepare new entry, with abstract
    if row["acl_id"] not in all_papers:
        all_papers[row["acl_id"]] = {
            "text": "",
            "abstract": row["abstract"]
        }
    
    # Add title and abstract
    all_papers[row["acl_id"]]["text"] += f"Title: {row['title']}\n\n"
    all_papers[row["acl_id"]]["text"] += f"ABSTRACT\n{row['abstract']}\n\n"

    text_content = parse_sections(row["json_contents"]["pdf_parse"]["body_text"])
    all_papers[row["acl_id"]]["text"] += text_content

R13-1042
{'1': 'SECTION 1: Introduction\n\nLaw enforcement agencies frequently obtain large amounts of electronic messages, such as emails, which they must search for evidence. However, individual messages may be useless without the conversational context they occur in. Most modern emails contain useful metadata such as the MIME header In-Reply-To, which marks relations between emails in a thread and can be used to disentangle threads. However, there are easy methods of obfuscating email threads: opening an email account for a single purpose; using multiple email accounts for one person; sharing one email account among multiple persons; changing the Subject header; and removing quoted material from earlier in the thread.\nHow can emails be organized by thread without metadata such as their MIME headers?\nWe propose to use text similarity metrics to identify emails belonging to the same thread. In this paper, as a first step for temporal thread disentanglement, we perform pairwise class

IndexError: list index out of range

In [93]:
all_papers["R13-1042"]["text"]

'Title: Headerless, Quoteless, but not Hopeless? Using Pairwise Email Classification to Disentangle Email Threads\n\nABSTRACT\nThread disentanglement is the task of separating out conversations whose thread structure is implicit, distorted, or lost. In this paper, we perform email thread disentanglement through pairwise classification, using text similarity measures on non-quoted texts in emails. We show that i) content text similarity metrics outperform style and structure text similarity metrics in both a class-balanced and class-imbalanced setting, and ii) although feature performance is dependent on the semantic similarity of the corpus, content features are still effective even when controlling for semantic similarity. We make available the Enron Threads Corpus, a newly-extracted corpus of 70,178 multiemail threads with emails from the Enron Email Corpus.\n\nSECTION 1: Introduction\n\nLaw enforcement agencies frequently obtain large amounts of electronic messages, such as emails, 